[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Boolean Masks &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Count Tromso's hours above 10 degrees, row 3.

In [2]:
print((readings[3] > 10).sum())

1847


The comparison makes the mask, and its sum counts the `True` values.

**2.** Print the mean of Oslo's readings that were below freezing, rounded to two places.

In [3]:
oslo = readings[1]

print(round(oslo[oslo < 0].mean(), 2))

-2.54


Filter first, then take the mean of what is left.

**3.** Count Bergen's hours from 10 degrees up to 15, using `&`.

In [4]:
bergen = readings[0]

print(((bergen >= 10) & (bergen < 15)).sum())

2095


Each comparison in its own parentheses, then `&`.

**4.** Make a mask of the hours in December from `np.arange(HOURS)` and `hour_of_year`, and print Svalbard's December mean, rounded to two places.

In [5]:
hours = np.arange(HOURS)
december = hours >= hour_of_year(12, 1)

print(december.sum())
print(round(readings[2, december].mean(), 2))

744
-12.06


December runs to the end of the year, so one comparison is enough. Svalbard has no missing hours in December, so `.mean()` needs no `nan` version here.

**5.** Copy Oslo's readings, replace every reading above 18 with 18, and print the largest value before and after.

In [6]:
capped = readings[1].copy()
capped[capped > 18] = 18

print(readings[1].max(), capped.max())

19.3 18.0


Oslo has no missing hours, so `.max()` works without its `nan` version.

---

&#8592; **Back to:** [Boolean Masks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/06-boolean-masks.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
